In [3]:

install.packages(c("naniar", "skimr", "ggplot2"))

library(naniar)
library(skimr)
library(ggplot2)


Installing packages into ‘/usr/local/lib/R/site-library’
(as ‘lib’ is unspecified)

also installing the dependencies ‘gridExtra’, ‘plyr’, ‘norm’, ‘visdat’, ‘viridis’, ‘UpSetR’



Attaching package: ‘skimr’


The following object is masked from ‘package:naniar’:

    n_complete




In [4]:

heart_url <- paste0(
  "https://archive.ics.uci.edu/ml/machine-learning-databases/",
  "heart-disease/processed.cleveland.data"
)

heart_names <- c(
  "age", "sex", "cp", "trestbps", "chol", "fbs",
  "restecg", "thalach", "exang", "oldpeak", "slope",
  "ca", "thal", "num"
)

heart_data <- read.csv(
  heart_url,
  header = FALSE,
  col.names = heart_names,
  na.strings = c("?", "", "NA"),
  strip.white = TRUE
)

heart_data[] <- lapply(heart_data, function(x) {
  suppressWarnings(as.numeric(as.character(x)))
})

head(heart_data)
str(heart_data)


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,num
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,63,1,1,145,233,1,2,150,0,2.3,3,0,6,0
2,67,1,4,160,286,0,2,108,1,1.5,2,3,3,2
3,67,1,4,120,229,0,2,129,1,2.6,2,2,7,1
4,37,1,3,130,250,0,0,187,0,3.5,3,0,3,0
5,41,0,2,130,204,0,2,172,0,1.4,1,0,3,0
6,56,1,2,120,236,0,0,178,0,0.8,1,0,3,0


'data.frame':	303 obs. of  14 variables:
 $ age     : num  63 67 67 37 41 56 62 57 63 53 ...
 $ sex     : num  1 1 1 1 0 1 0 0 1 1 ...
 $ cp      : num  1 4 4 3 2 2 4 4 4 4 ...
 $ trestbps: num  145 160 120 130 130 120 140 120 130 140 ...
 $ chol    : num  233 286 229 250 204 236 268 354 254 203 ...
 $ fbs     : num  1 0 0 0 0 0 0 0 0 1 ...
 $ restecg : num  2 2 2 0 2 0 2 0 2 2 ...
 $ thalach : num  150 108 129 187 172 178 160 163 147 155 ...
 $ exang   : num  0 1 1 0 0 0 0 1 0 1 ...
 $ oldpeak : num  2.3 1.5 2.6 3.5 1.4 0.8 3.6 0.6 1.4 3.1 ...
 $ slope   : num  3 2 2 3 1 1 3 1 2 3 ...
 $ ca      : num  0 3 2 0 0 0 2 0 1 0 ...
 $ thal    : num  6 3 7 3 3 3 3 3 7 7 ...
 $ num     : num  0 2 1 0 0 0 3 0 2 1 ...


In [5]:

set.seed(123)

heart_data$trestbps[1] <- -15
heart_data$trestbps[2] <- NA
heart_data$trestbps[3] <- 325
heart_data$trestbps[4] <- 305

head(heart_data$trestbps, 10)


[1] -15  NA 325 305 130 120 140 120 130 140

In [6]:

clean_bp <- function(bp) {
  if (length(bp) != 1) {
    stop("Provide one BP value.")
  }

  if (is.na(bp) || !is.finite(bp)) {
    return(NA_real_)
  } else if (bp < 0) {
    return(NA_real_)
  } else if (bp > 250) {
    return(250)
  } else {
    return(as.numeric(bp))
  }
}

heart_data$trestbps <- vapply(
  heart_data$trestbps,
  clean_bp,
  numeric(1)
)

head(heart_data$trestbps)


[1]  NA  NA 250 250 130 120

In [7]:

safe_mean_bp <- function(bp) {
  tryCatch({
    valid <- bp[!is.na(bp) & is.finite(bp)]

    if (length(valid) == 0) {
      stop("No valid BP observations.")
    }

    mean(valid)
  }, error = function(e) {
    warning(conditionMessage(e))
    NA_real_
  })
}

safe_ratio <- function(chol, bp) {
  tryCatch({
    if (length(chol) != 1 || length(bp) != 1) {
      stop("Provide one value for each variable.")
    }

    if (is.na(chol) || is.na(bp) ||
        !is.finite(chol) || !is.finite(bp)) {
      stop("Missing or invalid input.")
    }

    if (bp <= 0 || chol < 0) {
      stop("Invalid denominator or cholesterol value.")
    }

    chol / bp
  }, error = function(e) {
    warning(paste("Ratio error:", conditionMessage(e)))
    NA_real_
  })
}

cat("Mean BP:", safe_mean_bp(heart_data$trestbps), "\n")

cat("Cholesterol/BP ratio:",
    safe_ratio(heart_data$chol[1],
               heart_data$trestbps[1]), "\n")

safe_ratio(200, 0)
safe_ratio(200, NA_real_)


Mean BP: 132.3821 


Warning message in value[[3L]](cond):
“Ratio error: Missing or invalid input.”


Cholesterol/BP ratio: NA 


Warning message in value[[3L]](cond):
“Ratio error: Invalid denominator or cholesterol value.”


[1] NA

Warning message in value[[3L]](cond):
“Ratio error: Missing or invalid input.”


[1] NA

In [8]:

loop_clean <- function(bp) {
  result <- as.numeric(bp)

  for (i in seq_along(result)) {
    if (is.na(result[i]) || !is.finite(result[i])) {
      result[i] <- NA_real_
    } else if (result[i] < 0) {
      result[i] <- NA_real_
    } else if (result[i] > 250) {
      result[i] <- 250
    }
  }

  result
}

vector_clean <- function(bp) {
  result <- as.numeric(bp)

  result[!is.finite(result)] <- NA_real_
  result[!is.na(result) & result < 0] <- NA_real_
  result[!is.na(result) & result > 250] <- 250

  result
}

# Benchmark on a larger vector
bp_test <- rep(heart_data$trestbps, 1000)

loop_time <- system.time({
  loop_result <- loop_clean(bp_test)
})

vector_time <- system.time({
  vector_result <- vector_clean(bp_test)
})

print(loop_time)
print(vector_time)

cat(
  "Results identical:",
  isTRUE(all.equal(loop_result, vector_result)),
  "\n"
)

benchmark_results <- data.frame(
  Method = c("For loop", "Vectorized"),
  Elapsed_seconds = c(
    loop_time["elapsed"],
    vector_time["elapsed"]
  )
)

print(benchmark_results)
write.csv(
  benchmark_results,
  "benchmark_results.csv",
  row.names = FALSE
)


   user  system elapsed 
  0.173   0.005   0.179 
   user  system elapsed 
  0.007   0.006   0.013 
Results identical: TRUE 
      Method Elapsed_seconds
1   For loop           0.179
2 Vectorized           0.013


In [9]:

bp <- heart_data$trestbps

cat("Missing BP:", sum(is.na(bp)), "\n")
cat("Minimum BP:", min(bp, na.rm = TRUE), "\n")
cat("Maximum BP:", max(bp, na.rm = TRUE), "\n")
cat("Mean BP:", mean(bp, na.rm = TRUE), "\n")
cat("Median BP:", median(bp, na.rm = TRUE), "\n")

cat("Negative values remaining:",
    sum(bp < 0, na.rm = TRUE), "\n")

cat("Values above 250 remaining:",
    sum(bp > 250, na.rm = TRUE), "\n")

stopifnot(sum(bp < 0, na.rm = TRUE) == 0)
stopifnot(sum(bp > 250, na.rm = TRUE) == 0)

write.csv(
  heart_data,
  "cleaned_heart_data.csv",
  row.names = FALSE
)


Missing BP: 2 
Minimum BP: 94 
Maximum BP: 250 
Mean BP: 132.3821 
Median BP: 130 
Negative values remaining: 0 
Values above 250 remaining: 0 


In [10]:

adult_url <- paste0(
  "https://archive.ics.uci.edu/ml/machine-learning-databases/",
  "adult/adult.data"
)

adult_names <- c(
  "age", "workclass", "fnlwgt", "education",
  "education_num", "marital_status", "occupation",
  "relationship", "race", "sex", "capital_gain",
  "capital_loss", "hours_per_week", "native_country",
  "income"
)

adult_data <- read.csv(
  adult_url,
  header = FALSE,
  col.names = adult_names,
  na.strings = c("?", "NA"),
  strip.white = TRUE,
  stringsAsFactors = FALSE
)

adult_data[] <- lapply(adult_data, function(x) {
  if (is.character(x)) trimws(x) else x
})

# Introduce example data-quality problems
adult_data$age[1] <- NA_real_
adult_data$age[2] <- 999
adult_data$workclass[3] <- ""
adult_data$hours_per_week[4] <- NaN
adult_data$occupation[5] <- ""
adult_data$education_num[6] <- NA_real_

adult_before <- adult_data

head(adult_data)
str(adult_data)


,age,workclass,fnlwgt,education,education_num,marital_status,occupation,relationship,race,sex,capital_gain,capital_loss,hours_per_week,native_country,income
,<dbl>,<chr>,<int>,<chr>,<dbl>,<chr>,<chr>,<chr>,<chr>,<chr>,<int>,<int>,<dbl>,<chr>,<chr>
1,NA,State-gov,77516,Bachelors,13,Never-married,Adm-clerical,Not-in-family,White,Male,2174,0,40,United-States,<=50K
2,999,Self-emp-not-inc,83311,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,13,United-States,<=50K
3,38,,215646,HS-grad,9,Divorced,Handlers-cleaners,Not-in-family,White,Male,0,0,40,United-States,<=50K
4,53,Private,234721,11th,7,Married-civ-spouse,Handlers-cleaners,Husband,Black,Male,0,0,NaN,United-States,<=50K
5,28,Private,338409,Bachelors,13,Married-civ-spouse,,Wife,Black,Female,0,0,40,Cuba,<=50K
6,37,Private,284582,Masters,NA,Married-civ-spouse,Exec-managerial,Wife,White,Female,0,0,40,United-States,<=50K


'data.frame':	32561 obs. of  15 variables:
 $ age           : num  NA 999 38 53 28 37 49 52 31 42 ...
 $ workclass     : chr  "State-gov" "Self-emp-not-inc" "" "Private" ...
 $ fnlwgt        : int  77516 83311 215646 234721 338409 284582 160187 209642 45781 159449 ...
 $ education     : chr  "Bachelors" "Bachelors" "HS-grad" "11th" ...
 $ education_num : num  13 13 9 7 13 NA 5 9 14 13 ...
 $ marital_status: chr  "Never-married" "Married-civ-spouse" "Divorced" "Married-civ-spouse" ...
 $ occupation    : chr  "Adm-clerical" "Exec-managerial" "Handlers-cleaners" "Handlers-cleaners" ...
 $ relationship  : chr  "Not-in-family" "Husband" "Not-in-family" "Husband" ...
 $ race          : chr  "White" "White" "White" "Black" ...
 $ sex           : chr  "Male" "Male" "Male" "Male" ...
 $ capital_gain  : int  2174 0 0 0 0 0 0 0 14084 5178 ...
 $ capital_loss  : int  0 0 0 0 0 0 0 0 0 0 ...
 $ hours_per_week: num  40 13 40 NaN 40 40 16 45 50 40 ...
 $ native_country: chr  "United-States" "United-S

In [11]:

# NA detection
print(colSums(is.na(adult_data)))

# NaN detection
nan_counts <- vapply(adult_data, function(x) {
  if (is.numeric(x)) sum(is.nan(x)) else 0
}, numeric(1))

print(nan_counts)

# NULL demonstration
example_object <- NULL
print(is.null(example_object))

example_vector <- c(10, NA, 20)
print(is.null(example_vector))

# NULL removes a data-frame column when assigned to it
demo_df <- data.frame(x = 1:3, temporary = 4:6)
demo_df$temporary <- NULL
print(names(demo_df))

# Blank-string detection
for (column in names(adult_data)) {
  if (is.character(adult_data[[column]])) {
    cat(
      column, ":",
      sum(adult_data[[column]] == "", na.rm = TRUE),
      "blank values\n"
    )
  }
}

# Detect impossible age
print(adult_data[
  !is.na(adult_data$age) & adult_data$age == 999,
])


           age      workclass         fnlwgt      education  education_num 
             1           1836              0              0              1 
marital_status     occupation   relationship           race            sex 
             0           1843              0              0              0 
  capital_gain   capital_loss hours_per_week native_country         income 
             0              0              1            583              0 
           age      workclass         fnlwgt      education  education_num 
             0              0              0              0              0 
marital_status     occupation   relationship           race            sex 
             0              0              0              0              0 
  capital_gain   capital_loss hours_per_week native_country         income 
             0              0              1              0              0 
[1] TRUE
[1] FALSE
[1] "x"
workclass : 1 blank values
education : 0 blank values
marital

In [12]:

missing_before <- naniar::miss_var_summary(adult_before)
print(missing_before)

png("lab4_missingness_before.png",
    width = 1200, height = 800)

print(
  naniar::vis_miss(adult_before) +
    ggtitle("Missing Data Before Cleaning")
)

dev.off()


# A tibble: 15 × 3
   variable       n_miss pct_miss
   <chr>           <int>    <num>
 1 occupation       1843  5.66   
 2 workclass        1836  5.64   
 3 native_country    583  1.79   
 4 age                 1  0.00307
 5 education_num       1  0.00307
 6 hours_per_week      1  0.00307
 7 fnlwgt              0  0      
 8 education           0  0      
 9 marital_status      0  0      
10 relationship        0  0      
11 race                0  0      
12 sex                 0  0      
13 capital_gain        0  0      
14 capital_loss        0  0      
15 income              0  0      


agg_record_2be34e75698 
                     2

In [13]:

# Convert impossible ages to NA
adult_data$age[
  !is.na(adult_data$age) &
    (adult_data$age < 0 | adult_data$age > 120)
] <- NA_real_

# Replace blank categorical values with Unknown
for (column in names(adult_data)) {
  if (is.character(adult_data[[column]])) {
    blank_index <- !is.na(adult_data[[column]]) &
      trimws(adult_data[[column]]) == ""

    adult_data[[column]][blank_index] <- "Unknown"
  }
}

# Remove rows containing NaN in numeric columns
numeric_columns <- names(adult_data)[
  vapply(adult_data, is.numeric, logical(1))
]

nan_rows <- rep(FALSE, nrow(adult_data))

for (column in numeric_columns) {
  nan_rows <- nan_rows | is.nan(adult_data[[column]])
}

cat("Rows removed due to NaN:", sum(nan_rows), "\n")

adult_data <- adult_data[!nan_rows, , drop = FALSE]


Rows removed due to NaN: 1 


In [14]:

median_impute <- function(x) {
  if (!is.numeric(x)) {
    stop("Input must be numeric.")
  }

  x[is.nan(x)] <- NA_real_

  valid_values <- x[!is.na(x) & is.finite(x)]

  if (length(valid_values) == 0) {
    warning("No valid values available for imputation.")
    return(x)
  }

  median_value <- median(valid_values)

  x[is.na(x) | !is.finite(x)] <- median_value

  return(x)
}

# Test the function
test_values <- c(20, 30, NA, 40, NaN)
print(median_impute(test_values))

# Impute missing values in every numeric column
numeric_columns <- names(adult_data)[
  vapply(adult_data, is.numeric, logical(1))
]

for (column in numeric_columns) {
  adult_data[[column]] <- median_impute(adult_data[[column]])
}


[1] 20 30 30 40 30


In [15]:

complete_index <- complete.cases(adult_data)

cat("Complete rows:", sum(complete_index), "\n")
cat("Incomplete rows:", sum(!complete_index), "\n")

missing_summary <- function(data, stage) {
  total <- length(data)
  missing <- sum(is.na(data))

  data.frame(
    Stage = stage,
    Total_values = total,
    Missing_values = missing,
    Missing_percentage = round(100 * missing / total, 3)
  )
}

comparison <- rbind(
  missing_summary(adult_before, "Before cleaning"),
  missing_summary(adult_data, "After cleaning")
)

print(comparison)

print(naniar::miss_var_summary(adult_data))

write.csv(
  comparison,
  "missingness_before_after.csv",
  row.names = FALSE
)

png("lab4_missingness_after.png",
    width = 1200, height = 800)

print(
  naniar::vis_miss(adult_data) +
    ggtitle("Missing Data After Cleaning")
)

dev.off()


Complete rows: 30161 
Incomplete rows: 2399 
            Stage Total_values Missing_values Missing_percentage
1 Before cleaning           15           4265           28433.33
2  After cleaning           15           4262           28413.33
# A tibble: 15 × 3
   variable       n_miss pct_miss
   <chr>           <int>    <num>
 1 occupation       1843     5.66
 2 workclass        1836     5.64
 3 native_country    583     1.79
 4 age                 0     0   
 5 fnlwgt              0     0   
 6 education           0     0   
 7 education_num       0     0   
 8 marital_status      0     0   
 9 relationship        0     0   
10 race                0     0   
11 sex                 0     0   
12 capital_gain        0     0   
13 capital_loss        0     0   
14 hours_per_week      0     0   
15 income              0     0   


agg_record_2be6b9ebd03 
                     2

In [16]:

# Comprehensive summary
print(skimr::skim(adult_data))

# Validate impossible age values
invalid_age_count <- sum(
  adult_data$age < 0 | adult_data$age > 120,
  na.rm = TRUE
)

cat("Invalid age values remaining:", invalid_age_count, "\n")

# Validate numeric missing values
numeric_missing <- vapply(
  adult_data[numeric_columns],
  function(x) sum(is.na(x) | is.nan(x)),
  numeric(1)
)

print(numeric_missing)

# Validate blank categorical values
blank_remaining <- vapply(adult_data, function(x) {
  if (is.character(x)) {
    sum(x == "", na.rm = TRUE)
  } else {
    0
  }
}, numeric(1))

print(blank_remaining)

stopifnot(invalid_age_count == 0)
stopifnot(all(numeric_missing == 0))
stopifnot(all(blank_remaining == 0))

# Export cleaned dataset
write.csv(
  adult_data,
  "cleaned_adult_data.csv",
  row.names = FALSE
)

cat("Lab 4 completed successfully.\n")


── Data Summary ────────────────────────
                           Values    
Name                       adult_data
Number of rows             32560     
Number of columns          15        
_______________________              
Column type frequency:               
  character                9         
  numeric                  6         
________________________             
Group variables            None      

── Variable type: character ────────────────────────────────────────────────────
  skim_variable  n_missing complete_rate min max empty n_unique whitespace
1 workclass           1836         0.944   7  16     0        9          0
2 education              0         1       3  12     0       16          0
3 marital_status         0         1       7  21     0        7          0
4 occupation          1843         0.943   5  17     0       15          0
5 relationship           0         1       4  14     0        6          0
6 race                   0         1       5  1